In [6]:
from pathlib import Path

import torch
from torch_geometric.loader import DataLoader

from gnn_model import GINERegressor, smiles_to_data

In [7]:
MODEL_PATH = Path("models/gine_egfr_chembl203.pt")

print(MODEL_PATH.resolve())
print("Exists:", MODEL_PATH.exists())

C:\Users\dell\Documents\warsztaty_sztucznej_inteligencji\models\gine_egfr_chembl203.pt
Exists: True


In [8]:
def load_gine_model(model_path, device=None):
    if device is None:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    checkpoint = torch.load(model_path, map_location=device)

    model = GINERegressor(
        input_dim=checkpoint["input_dim"],
        edge_dim=checkpoint["edge_dim"],
        hidden_dim=checkpoint["hidden_dim"],
        num_layers=checkpoint["num_layers"],
        dropout=checkpoint["dropout"],
        pooling=checkpoint["pooling"],
        batch_norm=checkpoint["batch_norm"],
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model.to(device)
    model.eval()

    return model, checkpoint, device


model, checkpoint, device = load_gine_model(MODEL_PATH)

print("Loaded model")
print("Device:", device)
print("Target:", checkpoint["target_name"])
print("Test R²:", checkpoint["test_r2"])

Loaded model
Device: cpu
Target: Epidermal growth factor receptor
Test R²: 0.5435148013267626


In [9]:
@torch.no_grad()
def predict_pic50(smiles: str) -> float:
    data = smiles_to_data(smiles)

    if data is None:
        raise ValueError("Invalid SMILES or molecule could not be featurized.")

    loader = DataLoader([data], batch_size=1)
    batch = next(iter(loader)).to(device)

    pred = model(batch).view(-1).item()
    return float(pred)

In [10]:
smiles = "CCOc1ccc2nc(S(N)(=O)=O)sc2c1"

pred = predict_pic50(smiles)

print("SMILES:", smiles)
print("Target:", checkpoint["target_name"])
print("Predicted pIC50:", pred)

SMILES: CCOc1ccc2nc(S(N)(=O)=O)sc2c1
Target: Epidermal growth factor receptor
Predicted pIC50: 4.869700908660889
